# Module 10: Selection on the Outcome

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

The five agencies in this study were chosen because they had the **highest
use of force rates in the state**. The outcome being studied is the use of
force rate.

That is the most damaging pattern in program evaluation, and this module
measures exactly how much damage it does by running the same analysis on a
program that does not exist.

**About 25 minutes.**

## 1. Setup

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

GITHUB = "https://raw.githubusercontent.com/YinZhangCISER/Public-Safety-Statistics-Tutorials/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

monthly = pd.read_csv(BASE + "agency_monthly.csv")
profile = pd.read_csv(BASE + "agency_profile.csv")

# The five agencies that adopted the de escalation training in July 2023.
TRAINED = ["A001", "A002", "A004", "A007", "A010"]
TRUTH = -12.0                       # the effect built into the data, in percent

f = monthly[monthly["provisional"] == 0].copy()          # drop the unfinished months
f = f[~((f["agency_id"] == "A002") & (f["year_month"] == "2021-06"))]   # documented unrest
f["trained"] = f["agency_id"].isin(TRAINED).astype(int)

# Three periods, not two. The program phased in between July and November 2023.
f["period"] = np.where(f["year_month"] >= "2023-11", "after",
                       np.where(f["year_month"] < "2023-07", "before", "phase"))

NAME = dict(zip(profile["agency_id"], profile["agency_name"]))
COMPARISON = sorted(a for a in f["agency_id"].unique() if a not in TRAINED)


def rate(d):
    """Use of force per 100 arrests, pooled over whatever rows are passed in."""
    return 100 * d["n_uof"].sum() / d["n_arrests"].sum()


def cell_rate(agencies, period):
    return rate(f[f["agency_id"].isin(agencies) & (f["period"] == period)])


print(f"{f['agency_id'].nunique()} agencies, {f['year_month'].nunique()} months")
print(f"trained: {', '.join(NAME[a].split()[0] for a in TRAINED)}")

In [ ]:
keep = [a for a in TRAINED if a != "A007"]
base = profile.set_index("agency_id")["pre_program_uof_per_100_arrests"]

print("  every agency, ranked by its use of force rate before the program\n")
for a in sorted(base.index, key=lambda x: -base[x]):
    mark = "  <- selected" if a in TRAINED else ""
    print(f"    {base[a]:.3f}   {NAME[a]:34s}{mark}")

Four of the top five were selected, and the fifth selected agency sits sixth.
The rule was close to "take the worst five" and it was applied to the outcome
itself.

## 2. How unbalanced is that

Compare the two groups on everything the dataset records.

In [ ]:
prof = profile.set_index("agency_id")
cols = {"sworn_officers": "sworn officers",
        "population_served": "population served",
        "violent_crime_rate_per_1000": "violent crime rate",
        "property_crime_rate_per_1000": "property crime rate",
        "budget_share_public_safety_pct": "public safety budget share",
        "pre_program_uof_per_100_arrests": "USE OF FORCE RATE BEFORE"}
rows = []
for c, lab in cols.items():
    t = prof.loc[TRAINED, c].mean()
    u = prof.loc[COMPARISON, c].mean()
    rows.append({"characteristic": lab, "trained": round(t, 2),
                 "not trained": round(u, 2), "ratio": round(t / u, 2)})
pd.DataFrame(rows).set_index("characteristic")

Five characteristics within 15 percent of each other, and the sixth at 1.43.

**A balance table that stopped at the first five rows would have been
reassuring and wrong.** The variable that decided selection is the outcome's
own history, and it is the one most often left out.

## 3. Measuring the bias directly

The cleanest way to measure what selection does is to apply the same
selection rule to agencies that received **nothing**, and see what the
analysis reports.

In [ ]:
ranked = sorted(COMPARISON, key=lambda a: -base[a])
worst, best = ranked[:3], ranked[-3:]


def did(t_ids, c_ids):
    tb, ta = cell_rate(t_ids, "before"), cell_rate(t_ids, "after")
    cb, ca = cell_rate(c_ids, "before"), cell_rate(c_ids, "after")
    return 100 * ((ta / tb) / (ca / cb) - 1)


rows = [
    {"what was analysed": "the real program, before and after only",
     "estimate": f"{100 * (cell_rate(keep, 'after') / cell_rate(keep, 'before') - 1):+.1f}%",
     "true answer": "-12.0%"},
    {"what was analysed": "the real program, difference in differences",
     "estimate": f"{did(keep, COMPARISON):+.1f}%", "true answer": "-12.0%"},
    {"what was analysed": "NO program, given to the worst three untrained",
     "estimate": f"{did(worst, [a for a in COMPARISON if a not in worst]):+.1f}%",
     "true answer": "0.0%"},
    {"what was analysed": "NO program, given to the best three untrained",
     "estimate": f"{did(best, [a for a in COMPARISON if a not in best]):+.1f}%",
     "true answer": "0.0%"},
]
print(f"  worst three: {', '.join(NAME[a].split()[0] for a in worst)}")
print(f"  best three:  {', '.join(NAME[a].split()[0] for a in best)}\n")
pd.DataFrame(rows).set_index("what was analysed")

**A program that does not exist reports a 9.2 percent reduction** when given
to the worst three agencies, and a 10.5 percent **increase** when given to the
best three.

This is a correctly computed difference in differences, with a genuine
comparison group, on real records. The method is not at fault. The selection
rule is.

## 4. Why the real estimate survives

If selection manufactures 9 percent of bias, why does the real difference in
differences land on 12.5 against a truth of 12?

Because the bias is not a fixed quantity attached to the words "selected on
the outcome". It is whatever difference in **subsequent trajectory** the
selection created, and the parallel trends check in
[Module 7](Module_07_Testing_Parallel_Trends.ipynb) is what detects it.

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

f["lo"] = np.log(f["n_arrests"])
pi = pd.PeriodIndex(f["year_month"], freq="M")
f["yr"] = pi.year.values + (pi.month.values - 1) / 12.0
pre = f[f["period"] == "before"]

def pre_trend(ids):
    s = pre[pre["agency_id"].isin(ids)]
    z = smf.glm("n_uof ~ yr", s, family=sm.families.Poisson(),
                offset=s["lo"]).fit()
    return 100 * (np.exp(z.params["yr"]) - 1)


print("  each comparison, and the pre program trend gap inside it\n")
print("  study                treated   its comparison      gap")
for label, t_ids, c_ids in [
        ("the real study", keep, COMPARISON),
        ("worst three placebo", worst, [a for a in COMPARISON if a not in worst]),
        ("best three placebo", best, [a for a in COMPARISON if a not in best])]:
    t, c = pre_trend(t_ids), pre_trend(c_ids)
    print(f"  {label:20s} {t:+6.2f}%   {c:+11.2f}%   {t - c:+8.2f}")

There it is, in the last column.

The real study's two groups differ in pre program trend by **0.71
percentage points a year**, which is nothing. The two placebos differ by
**3.29 and 6.81 points**, and those gaps are what the analysis dutifully
reports as effects of 9.2 and 10.5 percent.

**Selection on the outcome does not bias the estimate by itself. It creates
the conditions for bias, and the parallel trends check from
[Module 7](Module_07_Testing_Parallel_Trends.ipynb) is what tells you whether
the conditions turned into a problem.** Here the check passes for the real
study and would have failed for both placebos.

That is a more useful
statement than "never select on the outcome", which is advice nobody in a
funding agency can follow.

## 5. What to write

> *Agencies were selected for the program on the basis of their pre program
> use of force rate, which is the study's outcome. The treated agencies'
> pre program rate averaged 43 percent above the comparison agencies', while
> the two groups differ by less than 15 percent on every other recorded
> characteristic. Selection on the outcome creates a risk of regression to the
> mean; the pre program trends of the two groups differ by 0.70 percent a
> year with a 95 percent interval from 3.31 below to 1.97 above, which does
> not indicate a trajectory difference large enough to account for the
> estimate.*

## Exercise

The placebo in section 3 used three agencies. Run it at every possible group
size and see how the manufactured effect behaves.

In [ ]:
# Fill in the blank, then run.
RUN = None          # try True

if RUN:
    rows = []
    for k in [2, 3, 4]:
        w = ranked[:k]
        b = ranked[-k:]
        rows.append({"group size": k,
                     "worst k given a fake program":
                         f"{did(w, [a for a in COMPARISON if a not in w]):+.1f}%",
                     "best k given a fake program":
                         f"{did(b, [a for a in COMPARISON if a not in b]):+.1f}%"})
    print("  every one of these agencies received nothing. The truth is 0.0%\n")
    display(pd.DataFrame(rows).set_index("group size"))
else:
    print("Set RUN above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

```python
RUN = True
```

The manufactured effect is large at every group size and it does not settle
down as the groups grow, because with seven agencies total, growing the
treated group shrinks the comparison group by the same amount.

**Both halves of a placebo matter.** A fake program given to the worst two
agencies is compared against the best five; given to the worst four, against
the best three. The contrast is between opposite ends of the same small
ranking either way, which is exactly the structure that manufactures effects.

The practical reading is that a placebo test on a small number of units is
itself noisy, and a single placebo result should be reported with the same
caution as the main estimate. [Module 12](Module_12_Placebo_Tests.ipynb) runs
several.

</details>

---

**Next:** [Module 11: Confounders, Mediators and Colliders](Module_11_Confounders_Mediators_And_Colliders.ipynb).

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*